# 00 — Euclidean Distance (The Wrong but Useful Way)

The distance formula from algebra class works perfectly in two-dimensional Cartesian space. Latitude and longitude look like a two-dimensional Cartesian space. This notebook is about what happens when you apply the former to the latter — and why the result is wrong, how wrong it is, and when you can get away with it anyway.

## 1. The Distance Formula

Given two points in a flat plane, the straight-line distance between them is:

```text
d = √( (x₂ - x₁)² + (y₂ - y₁)² )
```

This is the Pythagorean theorem applied to the right triangle formed by the two points and their shared corner. It works because the axes are in consistent, uniform units — both in meters, both in pixels, both in whatever.

Mapping this onto coordinates:

```text
x  →  longitude
y  →  latitude
```

The formula becomes:

```python
import math
d = math.sqrt((lon2 - lon1)**2 + (lat2 - lat1)**2)
```

And you get a number. The question — which this notebook will make you earn — is: **a number in what units?**

## 2. Compute Distance Between Two Points

In [1]:
import math

# Two points in [lon, lat] order — GeoJSON convention
p1 = (-98.5, 33.8)   # near Wichita Falls, TX
p2 = (-97.2, 34.1)   # about 130 km to the northeast

dx = p2[0] - p1[0]   # longitude difference
dy = p2[1] - p1[1]   # latitude difference

d = math.sqrt(dx**2 + dy**2)

print(f"p1: {p1}")
print(f"p2: {p2}")
print(f"dx (Δ lon): {dx}")
print(f"dy (Δ lat): {dy}")
print(f"d:  {d:.4f}")

p1: (-98.5, 33.8)
p2: (-97.2, 34.1)
dx (Δ lon): 1.2999999999999972
dy (Δ lat): 0.30000000000000426
d:  1.3342


## 3. What Are the Units?

You just computed `d ≈ 1.334`.

**1.334 what?**

Not kilometers. Not miles. Not meters.

**Degrees.**

`dx` and `dy` are differences in degrees of longitude and latitude. Squaring them, adding them, and taking the square root gives you a number in degrees — a hybrid angular unit that combines east-west and north-south angle into a single diagonal measurement.

That is not a physically meaningful distance. A degree of longitude at 33°N covers about 93 km. A degree of latitude covers about 111 km. They are different lengths. Treating them as equivalent axes — which is exactly what the Euclidean formula does — introduces an immediate geometric error.

The actual distance between these two points is roughly **133 km**. The Euclidean formula gave `1.334` — which is off by a factor of ~100, and the factor depends entirely on where on Earth you are.

In [2]:
# Wrap it as a reusable function
def euclidean_distance(p1, p2):
    """
    Straight-line distance between two [lon, lat] points.
    Returns a value in degrees — NOT kilometers.
    Use only for relative comparisons within small, consistent areas.
    """
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.sqrt(dx**2 + dy**2)


print(f"Euclidean distance: {euclidean_distance(p1, p2):.4f}°")
print("(Actual distance is ~133 km — the degree value is not comparable to km)")

Euclidean distance: 1.3342°
(Actual distance is ~133 km — the degree value is not comparable to km)


## 4. Visualize on a Map

Plotting both points with a line between them makes the geometry concrete. It also makes it obvious that the "distance" is a diagonal across the coordinate grid — not a physical path across the ground.

In [3]:
from ipyleaflet import Map, GeoJSON

points_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p1)}, "properties": {"name": "p1"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p2)}, "properties": {"name": "p2"}},
    ]
}

line_fc = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [list(p1), list(p2)]},
        "properties": {"label": f"d = {euclidean_distance(p1, p2):.4f}°  (not km)"}
    }]
}

center_lat = (p1[1] + p2[1]) / 2
center_lon = (p1[0] + p2[0]) / 2

m = Map(center=(center_lat, center_lon), zoom=9)
m.add(GeoJSON(data=points_fc))
m.add(GeoJSON(data=line_fc, style={"color": "#e63946", "weight": 2}))
m

Map(center=[33.95, -97.85], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## 5. When This Works

Euclidean distance on raw degrees is not entirely useless. Within a small, geographically consistent area, the degree-to-km ratio is approximately constant across all the points in your dataset — so the *relative ordering* of distances is roughly preserved even if the *absolute values* are wrong.

**Acceptable use cases:**
- Finding the nearest point out of a local set (same city, same county)
- Rough filtering — "which features are in the same neighborhood?"
- Sorting by proximity when you only care about rank, not exact distance
- Quick checks during development before switching to proper formulas

**Not acceptable:**
- Reporting an actual distance to a user
- Comparing distances across different latitudes
- Any computation where someone acts on the result
- Navigation, guidance, or range estimation of any kind

The test: *does the answer matter?* If yes, use Haversine. If you just need "roughly which direction is closer," Euclidean on degrees can hold the line temporarily.

---

## Exercise A — Distances Between Several Points

Compute the Euclidean distance from the base point to each of the targets below. Print the results sorted nearest to farthest.

In [4]:
import math

base = (-98.47, 33.91)   # Wichita Falls area

targets = [
    ("Tinker AFB",         (-97.37, 35.39)),
    ("NAS Fort Worth JRB", (-97.04, 32.85)),
    ("Lubbock",            (-101.87, 33.57)),
    ("Oklahoma City",      (-97.52, 35.47)),
    ("Abilene",            (-99.73, 32.45)),
]

# your code here
def euclidean_degrees(p1, p2):
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

results = sorted(
    [(name, euclidean_degrees(base, coord)) for name, coord in targets],
    key=lambda x: x[1]
)

print(f"{'Rank':<5} {'Name':<22} {'Euclidean (°)':>14}")
print("-" * 43)
for rank, (name, dist) in enumerate(results, 1):
    print(f"{rank:<5} {name:<22} {dist:>14.4f}")
# expected: sorted list of (name, euclidean_distance) from nearest to farthest

Rank  Name                    Euclidean (°)
-------------------------------------------
1     NAS Fort Worth JRB             1.7800
2     Oklahoma City                  1.8265
3     Tinker AFB                     1.8440
4     Abilene                        1.9285
5     Lubbock                        3.4170


## Exercise B — Rank Nearest Points

Using `euclidean_distance`, write a function `nearest_n(base, points, n)` that returns the `n` closest points from a list, sorted by distance. Test it on the targets above.

In [5]:
def nearest_n(base, named_points, n):
    """
    Returns the n closest (name, coord) pairs from named_points to base,
    sorted nearest first.
    named_points: list of (name, (lon, lat))
    """
    results = sorted(
        named_points,
        key = lambda item: euclidean_distance(base, item[1])
    )
    # your code here
    return results[:n]



top3 = nearest_n(base, targets, 3)
for name, coord in top3:
    print(f"{name}: {euclidean_distance(base, coord):.4f}°")

NAS Fort Worth JRB: 1.7800°
Oklahoma City: 1.8265°
Tinker AFB: 1.8440°


## Exercise C — Visualize All Distances

Plot `base` and all five targets on a map. Draw a line from `base` to each target. Label each line with the Euclidean distance value (in the `properties` dict — you can inspect it in the browser dev tools or just print it before displaying).

In [ ]:
# your code here
# build a FeatureCollection with:
#   - one Point feature for base
#   - one Point feature per target
#   - one LineString per target (base → target), with distance in properties
from ipyleaflet import Map, GeoJSON
from ipywidgets import VBox

def make_point(name, coord, role="target"):
    return {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": list(coord)},
        "properties": {"name": name, "role": role}
    }

def make_line(base, name, coord):
    dist = euclidean_degrees(base, coord)
    return {
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [list(base), list(coord)]
        },
        "properties": {
            "name":              name,
            "euclidean_degrees": round(dist, 4),
        }
    }

features = [make_point("Base (Wichita Falls)", base, role="base")]
for name, coord in targets:
    features.append(make_point(name, coord))
    features.append(make_line(base, name, coord))

fc = {"type": "FeatureCollection", "features": features}

# Print properties so they're inspectable without dev tools
print("LineString properties:")
for f in fc["features"]:
    if f["geometry"]["type"] == "LineString":
        p = f["properties"]
        print(f"  {p['name']:<22}  euclidean_degrees={p['euclidean_degrees']}")

line_layer = GeoJSON(
    data={
        "type": "FeatureCollection",
        "features": [f for f in features if f["geometry"]["type"] == "LineString"]
    },
    style={"color": "#e63946", "weight": 2, "opacity": 0.7}
)

point_layer = GeoJSON(
    data={
        "type": "FeatureCollection",
        "features": [f for f in features if f["geometry"]["type"] == "Point"]
    },
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.9, "weight": 2},
    point_style={"radius": 6}
)

all_coords  = [base] + [coord for _, coord in targets]
all_lons    = [c[0] for c in all_coords]
all_lats    = [c[1] for c in all_coords]
center      = (sum(all_lats) / len(all_lats), sum(all_lons) / len(all_lons))

m = Map(center=center, zoom=7)
m.add(line_layer)    # lines first so points render on top
m.add(point_layer)
m.fit_bounds([[min(all_lats), min(all_lons)], [max(all_lats), max(all_lons)]])

VBox([m])

LineString properties:
  Tinker AFB              euclidean_degrees=1.844
  NAS Fort Worth JRB      euclidean_degrees=1.78
  Lubbock                 euclidean_degrees=3.417
  Oklahoma City           euclidean_degrees=1.8265
  Abilene                 euclidean_degrees=1.9285


: 

---

## Check Your Understanding

You compute Euclidean distances from a base in Texas to two targets:

```python
base     = (-98.5, 33.8)
target_a = (-97.5, 33.8)   # 1.0° to the east, same latitude
target_b = (-98.5, 34.8)   # 1.0° to the north, same longitude
```

Both return `euclidean_distance = 1.0000`.

**Question:** Are these two targets actually the same distance from the base in the real world? If not, which is farther — and why?

```python
# your answer here
```


---

In [ ]:
# No — they are not the same real-world distance, even though both measure
# 1.0000° in Euclidean degree-space.
#
# A degree of LATITUDE is roughly constant everywhere: ~111 km.
# A degree of LONGITUDE shrinks with the cosine of latitude:
#   at 33.8°N:  cos(33.8°) ≈ 0.831  →  ~92 km per degree of longitude
#
# So:
#   target_b (1° north, same lon) is ~111 km away  — longer
#   target_a (1° east,  same lat) is  ~92 km away  — shorter
#
# target_b is farther by about 19 km — a ~20% real-world difference
# that euclidean_degrees reports as identical.

import math

base     = (-98.5, 33.8)
target_a = (-97.5, 33.8)
target_b = (-98.5, 34.8)

dist_a = haversine_km(base[1], base[0], target_a[1], target_a[0])
dist_b = haversine_km(base[1], base[0], target_b[1], target_b[0])

print(f"target_a (1° east):  {dist_a:.2f} km")
print(f"target_b (1° north): {dist_b:.2f} km")
print(f"difference:          {abs(dist_b - dist_a):.2f} km")
print(f"euclidean sees both as: 1.0000°")

# The flat-Earth model implicitly assumes a square degree grid —
# that one degree east equals one degree north in physical length.
# That is only true at the equator. Everywhere else, longitude degrees
# are shorter, and the error grows as you move toward the poles.

## Next

In [01 — Haversine Distance](./01-Haversine_Distance.ipynb), we replace the Euclidean formula with one that accounts for the curvature of the Earth — producing real distances in kilometers that hold up anywhere on the globe.